In [10]:
import warnings
import numpy as np
import pandas as pd
import pickle

from sklearn.preprocessing import StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.metrics.pairwise import euclidean_distances
from sklearn.metrics import silhouette_score

warnings.filterwarnings("ignore")

In [11]:
df = pd.read_csv("final_preprocessed_movies.csv")
df

,id,title,overview,poster_path,release_date,release_year,vote_average,vote_count,runtime,original_language,popularity,genres,budget,keywords,content,revenue,original_title
0,2,Ariel,After the coal mine he works at closes and his...,https://image.tmdb.org/t/p/w500/ojDg0PGvs6R9xY...,1988-10-21,1988,7.100,262,73,fi,8.155,"Drama, Comedy, Romance",0,"prison, underdog, helsinki, finland, factory w...","Ariel Drama, Comedy, Romance After the coal mi...",0,Ariel
1,3,Shadows in Paradise,"Nikander, a rubbish collector and would-be ent...",https://image.tmdb.org/t/p/w500/nj01hspawPof0m...,1986-10-17,1986,7.199,281,74,fi,5.946,"Drama, Comedy, Romance",0,"helsinki, finland, salesclerk, garbage","Shadows in Paradise Drama, Comedy, Romance Nik...",0,Varjoja paratiisissa
2,5,Four Rooms,It's Ted the Bellhop's first night on the job....,https://image.tmdb.org/t/p/w500/75aHn1NOYXh4M7...,1995-12-09,1995,5.784,2436,98,en,15.295,Comedy,4000000,"hotel, new year's eve, witch, bet, sperm, hote...",Four Rooms Comedy It's Ted the Bellhop's first...,4257354,Four Rooms
3,6,Judgment Night,"While racing to a boxing match, Frank, Mike, J...",https://image.tmdb.org/t/p/w500/3rvvpS9YPM5HB2...,1993-10-15,1993,6.533,302,109,en,13.564,"Action, Crime, Thriller",21000000,"drug dealer, chicago, illinois, escape, one ni...","Judgment Night Action, Crime, Thriller While r...",12136938,Judgment Night
4,11,Star Wars,Princess Leia is captured and held hostage by ...,https://image.tmdb.org/t/p/w500/6FfCtAuVAW8XJj...,1977-05-25,1977,8.204,19155,121,en,88.559,"Adventure, Action, Science Fiction",11000000,"empire, galaxy, rebellion, android, hermit, sm...","Star Wars Adventure, Action, Science Fiction P...",775398007,Star Wars
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
69400,1723522,Kaafi Thota,A story of a deceitful love story and a legal ...,/lGkyWRdUSirI6cSB8uHKmsmtTYf.jpg,2017-08-18,2017,0.000,0,130,kn,0.029,"Thriller, Romance",0,NaN,"Kaafi Thota Thriller, Romance A story of a dec...",0,ಕಾಫಿ ತೋಟ
69401,1723571,BB5,A dejected writer's creative skills misused by...,/asnrO2LqzFQQJE8Hd9KDi7aLGn2.jpg,2017-05-26,2017,0.000,0,123,kn,0.029,"Crime, Thriller, Mystery",0,NaN,"BB5 Crime, Thriller, Mystery A dejected writer...",0,BB5
69402,1723580,Ranganayaki,A story about a gang-rape survivor and her fig...,/9GsqPKcwjGfyo6d56asNQxp3QaD.jpg,2019-11-01,2019,0.000,0,117,kn,0.057,"Thriller, Crime",0,NaN,"Ranganayaki Thriller, Crime A story about a ga...",0,Ranganayaki
69403,1723626,B.I.G - A World First,"On September 20, 2023, Jakob Schubert was able...",https://image.tmdb.org/t/p/w500/gzjOuqUOYUeYKd...,2026-06-18,2026,7.000,1,54,en,0.479,"Adventure, Documentary",0,"cave, norway, free climbing, bouldering, climbing","B.I.G - A World First Adventure, Documentary O...",0,B.I.G - A World First


In [13]:
num = [
    "vote_average", "vote_count", "runtime",
    "popularity", "release_year", "budget"
]

genres = df["genres"].str.get_dummies(sep=", ")

languages = pd.get_dummies(
    df["original_language"].where(
        df["original_language"].isin(
            df["original_language"].value_counts().head(10).index
        ),
        "other"
    ) 
)

tfidf = TfidfVectorizer(max_features=100, stop_words="english")
content_features = tfidf.fit_transform(df["content"]).toarray()

In [14]:
num = StandardScaler().fit_transform(df[num])

In [15]:
X = np.concatenate([
    num,
    genres.values,
    languages.values,
    content_features
], axis=1)

print("Feature Matrix:", X.shape)

Feature Matrix: (69405, 136)


In [16]:
model_k = KMeans(
    n_clusters=10,
    random_state=42,
    n_init=10
)

df["cluster"] = model_k.fit_predict(X)

In [17]:
print("WCSS:", round(model_k.inertia_, 2))
print("Silhouette:", round(
    silhouette_score(X, df["cluster"]), 4
))

WCSS: 345274.73
Silhouette: 0.102


In [18]:
model_bundle = {"model": model_k, "df": df, "X": X, "numeric_columns": num, "genre_columns": genres.columns.tolist(), "language_columns": languages.columns.tolist(), "tfidf": tfidf}

In [19]:
with open('clustering_model.pkl','wb') as file:
    pickle.dump(model_bundle,file)

In [20]:
with open('clustering_model.pkl','rb') as file:
    model_bundle = pickle.load(file)

In [21]:
def recommend_movie(title,n=10):
    model = model_bundle["model"]
    df = model_bundle["df"]
    X = model_bundle["X"]

    movie = df[df["title"].astype(str).str.lower() == title.strip().lower()]
    if movie.empty:
        return None
    
    movie_index = movie.index[0]
    movie_features = X[movie_index].reshape(1,-1)
    prediction = model.predict(movie_features)
    cluster = int(prediction[0])
    cluster_indices = df[df["cluster"] == cluster].index

    distances = euclidean_distances(movie_features,X[cluster_indices])[0]
    result = pd.DataFrame({"index": cluster_indices, "distance": distances})
    result = result[result["index"] != movie_index]
    result = result.sort_values("distance")
    result = result.head(n)

    cols = [
        "id", "title", "original_title", "overview", "poster_path",
        "release_date", "release_year", "vote_average", "vote_count",
        "runtime", "genres", "original_language", "popularity", "budget", "revenue"
    ]
    available_cols = [c for c in cols if c in df.columns]
    rec_df = df.loc[result["index"], available_cols].copy()
    rec_df["similarity"] = [int(round(max(60, min(99, 100 - (d * 5))))) for d in result["distance"]]
    if "poster_path" in rec_df.columns:
        rec_df["poster_path"] = rec_df["poster_path"].apply(
            lambda x: f"https://image.tmdb.org/t/p/w500{x}" if isinstance(x, str) and x.startswith("/") else (x if isinstance(x, str) else "")
        )
    recommendations = rec_df.fillna("").to_dict(orient="records")

    queried_movie = df.loc[movie_index, available_cols].to_dict()
    if isinstance(queried_movie.get("poster_path"), str) and queried_movie["poster_path"].startswith("/"):
        queried_movie["poster_path"] = f"https://image.tmdb.org/t/p/w500{queried_movie['poster_path']}"

    return {
        "movie": queried_movie,
        "cluster": cluster,
        "recommendations": recommendations
    }

In [22]:
recommend_movie('Inception',10)

{'movie': {'id': 27205,
  'title': 'Inception',
  'original_title': 'Inception',
  'overview': 'Cobb, a skilled thief who commits corporate espionage by infiltrating the subconscious of his targets is offered a chance to regain his old life as payment for a task considered to be impossible: "inception", the implantation of another person\'s idea into a target\'s subconscious.',
  'poster_path': 'https://image.tmdb.org/t/p/w500/oYuLEt3zVCKq57qu2F8dT7NIa6f.jpg',
  'release_date': '2010-07-15',
  'release_year': 2010,
  'vote_average': 8.364,
  'vote_count': 34495,
  'runtime': 148,
  'genres': 'Action, Science Fiction, Adventure',
  'original_language': 'en',
  'popularity': 83.952,
  'budget': 160000000,
  'revenue': 825532764},
 'cluster': 3,
 'recommendations': [{'id': 157336,
   'title': 'Interstellar',
   'original_title': 'Interstellar',
   'overview': 'The adventures of a group of explorers who make use of a newly discovered wormhole to surpass the limitations on human space trave